# Two coupled antimony donor simulation

This notebook is a tutorial for the `Sb2.py` module.  The simulated system is one electron spin coupled to two `123Sb` nuclear spins in silicon:

$$\mathcal{H}=\mathcal{H}_e\otimes\mathcal{H}_{Sb_1}\otimes\mathcal{H}_{Sb_2},\qquad \dim(\mathcal{H})=2\times8\times8=128.$$

The model follows the high-spin donor Hamiltonian used for single antimony donors and extends it to two nuclei:

$$H=\gamma_e B_0 S_z-\gamma_{n,1}B_0 I_{1,z}-\gamma_{n,2}B_0 I_{2,z}+A_1\mathbf{S}\cdot\mathbf{I}_1+A_2\mathbf{S}\cdot\mathbf{I}_2+H_{Q,1}+H_{Q,2}+H_{1,2}.$$

`Sb2.py` also contains ideal unitary operations used in the nuclear-spin qudit literature: global SU(2) rotations, site-local SU(2) rotations, two-level Givens rotations, and SNAP phase gates.  Where a shared primitive already exists, `Sb2.py` delegates to functions in the main `psyduck` package rather than reimplementing it.

## 1. Imports and basis convention

The tensor order is fixed as `electron x Sb_1 x Sb_2`.  The full-space operators are:

- Electron: `Sx`, `Sy`, `Sz`
- First antimony nucleus: `I1x`, `I1y`, `I1z`
- Second antimony nucleus: `I2x`, `I2y`, `I2z`

The nuclear basis is ordered by QuTiP angular momentum convention: `m_I = +7/2, +5/2, ..., -7/2`.  The import cell below makes the repository root visible first, so both the local `Sb2.py` file and the shared `psyduck` package can be imported when this notebook is launched from the `examples/coupled antimony` directory.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

# Make the repo root importable when Jupyter starts in this notebook folder.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "psyduck").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from psyduck.hamiltonians import zeeman_hamiltonian
from psyduck.operations import global_rotation, parity_operator, snap

import Sb2

# Reload while editing Sb2.py from the same notebook session.
Sb2 = importlib.reload(Sb2)

print("Shared psyduck functions imported:", global_rotation.__name__, snap.__name__, parity_operator.__name__)
print(f"Full Hilbert-space dimension: {Sb2.DIM_FULL}")
print(f"Nuclear m_I basis: {Sb2.mI_values}")
print(f"Sx shape: {Sb2.Sx.shape}")
print(f"I1z shape: {Sb2.I1z.shape}")
print(f"I2z shape: {Sb2.I2z.shape}")

## 2. Build a two-antimony Hamiltonian

The two nuclei are allowed to have different hyperfine couplings and quadrupole splittings.  That is the simplest way to represent two physical antimony atoms at different electron-density and strain environments.  The optional `J_zz` term is a small secular nuclear-nuclear coupling.

In [ ]:
params = Sb2.Sb2Parameters(
    B=1.38,              # tesla
    A_1=101.52e-3,       # 101.52 MHz in GHz
    A_2=74.0e-3,         # second donor has weaker electron overlap
    fq_1=-44.1e-6,       # -44.1 kHz in GHz
    fq_2=-31.0e-6,       # different local EFG for the second donor
    J_zz=0.25e-6,        # 0.25 kHz secular coupling
)

H = Sb2.H_from_params(params)

print(f"H shape: {H.shape}")
print(f"H is Hermitian: {H.isherm}")
print(f"Lowest 8 energies relative to ground (MHz):")
print(np.round(Sb2.eigenenergies(H)[:8] * 1e3, 6))

## 3. Inspect the Hamiltonian terms

The electron Zeeman term is by far the largest scale.  The hyperfine terms split the nuclear transitions by tens of MHz depending on the electron spin.  The quadrupole and explicit nuclear-nuclear coupling terms are much smaller but break degeneracies and make adjacent transitions non-uniform.

In [ ]:
terms = {
    "electron Zeeman": Sb2.H_electron_zeeman(B=params.B, ge=params.gamma_electron),
    "nuclear Zeeman": Sb2.H_nuclear_zeeman(B=params.B, gn_1=params.gamma_n_1, gn_2=params.gamma_n_2),
    "hyperfine": Sb2.H_hyperfine(A_1=params.A_1, A_2=params.A_2),
    "quadrupole": Sb2.H_quadrupole(fq_1=params.fq_1, fq_2=params.fq_2),
    "nuclear-nuclear": Sb2.H_nuclear_nuclear(J_zz=params.J_zz),
}

electron_zeeman_from_psyduck = Sb2.embed_electron_operator(
    zeeman_hamiltonian(Sb2.S_ELECTRON, B0=params.B, gamma=-params.gamma_electron)
)
print(
    "Electron Zeeman wrapper matches psyduck.hamiltonians.zeeman_hamiltonian:",
    np.allclose(terms["electron Zeeman"].full(), electron_zeeman_from_psyduck.full()),
)
print()

for name, term in terms.items():
    # Spectral width is a useful scale estimate for a Hamiltonian term.
    ev = term.eigenenergies()
    print(f"{name:16s}: width = {(ev[-1] - ev[0]) * 1e3:10.6f} MHz")

## 4. Product-basis NMR transition frequencies

In the high-field limit, the eigenstates are close to `|m_s, m_1, m_2>` product states.  The helper below estimates adjacent `Delta m_I = 1` transition frequencies by taking expectation values in those product states.  This is fast and useful for intuition before doing full diagonalization or driven dynamics.

In [ ]:
Sb2.print_adjacent_nmr_frequencies(H, site="1", electron="down", other_m=-7/2, unit="MHz")
print()
Sb2.print_adjacent_nmr_frequencies(H, site="2", electron="down", other_m=-7/2, unit="MHz")
print()
Sb2.print_adjacent_nmr_frequencies(H, site="1", electron="up", other_m=-7/2, unit="MHz")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

for site, marker in [("1", "o"), ("2", "s")]:
    rows = Sb2.adjacent_nmr_frequencies(H, site=site, electron="down", other_m=-7/2)
    labels = [f"{a:g}->{b:g}" for a, b, _ in rows]
    freqs_mhz = [f * 1e3 for _, _, f in rows]
    ax.plot(labels, freqs_mhz, marker=marker, lw=1.5, label=f"Sb_{site}")

ax.set_ylabel("transition frequency (MHz)")
ax.set_xlabel("nuclear transition")
ax.legend()
ax.grid(alpha=0.25)
plt.xticks(rotation=35)
plt.tight_layout()

## 5. Sweep the second hyperfine coupling

This sweep mimics moving the electron wavefunction between the two donor sites or fitting a device where the electron has unequal density at the two nuclei.  Only the low-energy manifold is plotted here so the structure is visible.

In [ ]:
A_2_values = np.linspace(0.0, params.A_1, 25)
spectra_mhz = []

for A_2 in A_2_values:
    H_sweep = Sb2.H_total(
        B=params.B,
        A_1=params.A_1,
        A_2=A_2,
        fq_1=params.fq_1,
        fq_2=params.fq_2,
        J_zz=params.J_zz,
    )
    spectra_mhz.append(Sb2.eigenenergies(H_sweep)[:32] * 1e3)

spectra_mhz = np.array(spectra_mhz)

fig, ax = plt.subplots(figsize=(7, 4))
for level in range(spectra_mhz.shape[1]):
    ax.plot(A_2_values * 1e3, spectra_mhz[:, level], color="black", lw=0.8, alpha=0.45)

ax.set_xlabel("A_2 (MHz)")
ax.set_ylabel("energy relative to ground (MHz)")
ax.set_title("Low-energy spectrum while turning on the second hyperfine coupling")
ax.grid(alpha=0.2)
plt.tight_layout()

## 6. Global and local SU(2) rotations

The cat-state paper emphasizes global SU(2) rotations in the spin-7/2 qudit.  `Sb2.py` has both local rotations on one nucleus and global rotations acting on both nuclei.  The example below starts from `|down, -7/2, -7/2>` and applies a global `pi/2` rotation about `x`.

In [ ]:
psi0 = Sb2.product_state(electron="down", m_1=-7/2, m_2=-7/2)

U_global = Sb2.global_covariant_su2_rotation(theta=np.pi / 2, phi=0.0)
psi_global = U_global * psi0

U_local = Sb2.nuclear_rotation(site="1", angle=np.pi / 2, axis="x")
U_local_from_psyduck = Sb2.embed_nuclear_operator(
    "1", global_rotation(Sb2.I_SB, np.pi / 2, "x")
)
print(
    "Local rotation wrapper matches psyduck.operations.global_rotation:",
    np.allclose(U_local.full(), U_local_from_psyduck.full()),
)

psi_local = U_local * psi0

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
axes[0].bar(Sb2.mI_values, Sb2.nuclear_populations(psi_global, "1"), width=0.75)
axes[0].set_title("Sb_1 after global pi/2")
axes[1].bar(Sb2.mI_values, Sb2.nuclear_populations(psi_global, "2"), width=0.75)
axes[1].set_title("Sb_2 after global pi/2")

for ax in axes:
    ax.set_xlabel("m_I")
    ax.set_ylabel("population")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)

plt.tight_layout()

print("Local rotation only changes Sb_1:")
print("  <I1z> =", Sb2.iz_expectation(psi_local, "1"))
print("  <I2z> =", Sb2.iz_expectation(psi_local, "2"))

## 7. SNAP phase gates and one-axis twisting

A SNAP gate is diagonal in the `|m_I>` basis and applies programmable phases to each nuclear level.  In a generalized rotating frame this can be implemented virtually by redefining oscillator phases.  The `one_axis_twist` helper is a SNAP phase pattern proportional to `m_I^2`.

In [ ]:
phases_1 = 0.35 * Sb2.mI_values**2
phases_2 = -0.20 * Sb2.mI_values**2

U_snap = Sb2.two_nucleus_snap(phases_1=phases_1, phases_2=phases_2)
U_snap_1_from_psyduck = Sb2.embed_nuclear_operator(
    "1", snap(phases_1, dim=Sb2.DIM_NUCLEAR)
)
print(
    "Single-site SNAP wrapper matches psyduck.operations.snap:",
    np.allclose(Sb2.snap_gate("1", phases_1).full(), U_snap_1_from_psyduck.full()),
)

psi_snap = U_snap * psi_global

U_twist_1 = Sb2.one_axis_twist(site="1", chi=np.pi / 7)
psi_twist = U_twist_1 * psi_global

print(f"Parity before SNAP, Sb_1: {Sb2.parity_expectation(psi_global, '1'):+.6f}")
print(f"Parity after SNAP,  Sb_1: {Sb2.parity_expectation(psi_snap, '1'):+.6f}")
print(f"Parity after twist, Sb_1: {Sb2.parity_expectation(psi_twist, '1'):+.6f}")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(Sb2.mI_values, np.angle(np.exp(1j * phases_1)), "o-", label="Sb_1 SNAP phase")
ax.plot(Sb2.mI_values, np.angle(np.exp(1j * phases_2)), "s-", label="Sb_2 SNAP phase")
ax.set_xlabel("m_I")
ax.set_ylabel("wrapped phase (rad)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

## 8. Ideal nuclear cat states

`Sb2.py` includes ideal state constructors, which are useful targets for pulse design.  This example embeds an extreme spin-cat state on `Sb_1` while leaving `Sb_2` polarized.

In [ ]:
cat_1 = Sb2.naked_nuclear_cat_state(m_abs=7/2, phase=np.pi / 2)
psi_cat = Sb2.embed_nuclear_state(site="1", psi_nuclear=cat_1, electron="down", other_m=-7/2)

print("Sb_1 populations for ideal |+7/2> + i|-7/2> cat:")
print(np.round(Sb2.nuclear_populations(psi_cat, "1"), 6))
print("Sb_2 remains polarized:")
print(np.round(Sb2.nuclear_populations(psi_cat, "2"), 6))
print(f"Sb_1 parity: {Sb2.parity_expectation(psi_cat, '1'):+.6f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(Sb2.mI_values, Sb2.nuclear_populations(psi_cat, "1"), width=0.75)
ax.set_xlabel("m_I")
ax.set_ylabel("population")
ax.set_title("Ideal Sb_1 extreme cat populations")
ax.set_ylim(0, 0.6)
ax.grid(alpha=0.25)
plt.tight_layout()

## 9. Time evolution under the static Hamiltonian

`Sb2.evolve_state` accepts Hamiltonians in GHz and times in ns.  It internally multiplies the Hamiltonian by `2*pi` so QuTiP receives angular frequencies.  The example below follows a globally rotated state under the full static Hamiltonian and tracks `Iz` on both nuclei.

In [ ]:
times_ns = np.linspace(0, 250, 251)

result = Sb2.evolve_state(
    H,
    psi_global,
    times_ns,
    e_ops=[Sb2.I1z, Sb2.I2z, Sb2.Sz],
    options={"store_states": True},
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(times_ns, result.expect[0], label="<I1z>")
ax.plot(times_ns, result.expect[1], label="<I2z>")
ax.plot(times_ns, result.expect[2], label="<Sz>")
ax.set_xlabel("time (ns)")
ax.set_ylabel("expectation value")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

## 10. A simple conditional phase from the explicit nuclear-nuclear term

The optional `J_zz I1z I2z` term creates phases that depend on both nuclear spin projections.  The next cell exaggerates `J_zz` so the effect is visible in a short simulated time.  Use experimentally justified values for quantitative work.

In [ ]:
H_coupling_only = Sb2.H_nuclear_nuclear(J_zz=20e-6)  # 20 kHz, exaggerated for visibility

psi_1_plus = Sb2.naked_spin_coherent_state(theta=np.pi / 2, phi=0.0, reference_m=-7/2)
psi_2_plus = Sb2.naked_spin_coherent_state(theta=np.pi / 2, phi=0.0, reference_m=-7/2)
psi_plus_plus = Sb2.two_nucleus_product(psi_1_plus, psi_2_plus, electron="down")

coupling_times_ns = np.linspace(0, 5_000, 101)
coupling_result = Sb2.evolve_state(
    H_coupling_only,
    psi_plus_plus,
    coupling_times_ns,
    e_ops=[
        Sb2.embed_nuclear_operator("1", parity_operator(Sb2.I_SB)),
        Sb2.embed_nuclear_operator("2", parity_operator(Sb2.I_SB)),
    ],
    options={"store_states": True},
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(coupling_times_ns / 1_000, coupling_result.expect[0], label="Sb_1 parity")
ax.plot(coupling_times_ns / 1_000, coupling_result.expect[1], label="Sb_2 parity")
ax.set_xlabel("time (us)")
ax.set_ylabel("parity")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

## 11. Suggested extensions

- Replace the axial quadrupole terms with full `3 x 3` tensors using `Sb2.H_quadrupole_tensor` or the `Qab_1` and `Qab_2` arguments of `Sb2.H_total`.
- Add time-dependent RF tones with `Sb2.H_nmr_drive` or `Sb2.H_global_nmr_drive` and QuTiP coefficient functions.
- Fit `A_1`, `A_2`, `fq_1`, `fq_2`, and `J_zz` against measured transition frequencies.
- Use the ideal gates here as targets for GRAPE, CRAB, or hand-built multi-tone pulse optimization.